In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

c:\Users\STUDENT\Downloads\land_monitor\land-monitor


In [ ]:
from app.services.predictor import LandUsePredictor
from ml.data import  load_eurosat, load_stats
from ml.splits import load_splits
from pathlib import Path
from ml.transform import eval_transforms
import numpy as np
import torch_process

In [9]:
predictor = LandUsePredictor(Path("models/model.onnx"), Path("models/eurosat_stats.json"), Path("models/class_names.json"))

In [10]:
ds=load_eurosat("data/EuroSAT")
test_idx=load_splits("configs/eurosat_splits.json")["test"]

stats=load_stats(Path("models/eurosat_stats.json"))
torch_transform=eval_transforms(stats["mean"], stats["std"])

print("Class names match:", predictor.class_names == ds.classes)

Class names match: True


In [12]:
rng = np.random.default_rng(seed=0)
sample = rng.choice(test_idx, size=5, replace=False)
print(sample)


[17194 13815  7340  8296 22953]


In [16]:
for idx in sample:
    image, label = ds[int(idx)]

    api_input = predictor.preprocess(image)[0]
    torch_input = torch_transform(image).numpy()
    same = np.allclose(api_input, torch_input, atol=1e-6)

    top_class, confidence = predictor.predict(image)[0]
    true_class = ds.classes[label]

    print(
        f"image {idx}: preprocessing match={same} | "
        f"true={true_class} | predicted={top_class} ({confidence:.1%})"
    )

TypeError: Image.convert() got an unexpected keyword argument 'dtype'